<img src="images/Logo_UCLL_ENG_RGB.png" style="background-color:white;" />

AI Applications
# Computer Vision , Convolutional networks

Lecturer: Jan D'Espallier

Inspired (a.o) by: lecture from Chidi Nweke 2022-2023 & Keras tutorials

Academic year 2027-2028

Our goal for this class is that by the end of it you 

- Will have a feeling of how computer vision was done in the past. Engineered feature extraction, used (for example) in image classification.

- Will understand the power of CNN and you will be able to read a diagram like the one below. Also you should be able to design and train one to use it for image recognition tasks.

<center><img src="images/cnn architecture.webp" style="background-color:white; max-width:50%"/></center>

- Will understand that these days you do not have to design and train your own cnn architecture anymore. Instead you better reuse pre-designed/trained networks, and fine tune them for your application.

## How this notebook works..
In this notebook you'll find cells that start with a ❓. These cells contain a question or a task.
The text in the cell contains at the end an indication of the type of content that is expected in the next cell.
- 'txt' : the next cell is a 'mark down' cell, in which you should write in text , your answer to the question.
- 'code': the next cell is a 'code' cell, in which you should write your code an execute it.
- 'exec' : there is no next cell, just execute the task.
Do not remove the answer-cells behind the task-cells. The cells are needed as they exist to evaluate your submission.

Lets start with an easy one:

❓ Write in the cell below your ucll email adress. (txt)

**Name:** [enter your first and last name]

**UCLL email:** [enter your UCLL email address]

## Getting your environment ready
This notebook has been tested in an anaconda environment on windows with python 3.11.9 installed.

It should run as well on Linux and or with other versions of python as long as the correct packages have been installed.
pip install -r requirements should do the job

Installing the packages manually is also an option.
 - pip install jupyter
 - pip install matplotlib
 - pip install scikit-learn
 - pip install seaborn

some new packages since the notebook on neural nets :
 - pip install tensorflow
 - pip install tensorflow-datasets
 - pip install opencv-python
 - pip install scikit-image
  

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score
import tensorflow as tf
import keras
import tensorflow_datasets as tfds
import pathlib
import numpy as np
import cv2 as cv

## Previous course
This notebook is a following the one on neural networks. It is advised that you first run trough that notebook to get a good understanding of some basic concepts which are reused in this notebook.
The following 3 functions are also copy/pasted from that notebook.

In [ ]:
def plotTrainingHistory(history):
    # Create a figure and a set of subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))  # Adjusted figsize to better fit two subplots
    
    # Plot training and validation loss on the first subplot
    ax1.plot(history.history['loss'], label='train')
    ax1.plot(history.history['val_loss'], label='val')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    # Plot training and validation accuracy on the second subplot
    ax2.plot(history.history['accuracy'], label='train')
    ax2.plot(history.history['val_accuracy'], label='val')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    
    # Improve layout to prevent overlap
    plt.tight_layout()
    
    # Display the plot
    plt.show()

In [ ]:
def displayPerformanceFigures(real,pred,title,includeCF=True):
   if includeCF:
       # Compute confusion matrix
       cm = confusion_matrix(real, pred)
       sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
       plt.xlabel('Predicted labels')
       plt.ylabel('True labels')
       plt.title('Confusion Matrix for '+title)
       plt.show() 

   acc= accuracy_score(real, pred)
   f1 = f1_score(real, pred, average='macro')

   performance = {
        "acc": round(acc, 3),
        "f1": round(f1, 3),
   }
   print(performance)

In [ ]:
def compile_and_train_model(mod,nbepochs=25,pat=3):
    mod.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience=pat,restore_best_weights=True) # If the validation loss rises for 3 epochs training is stopped..and the weigths of the model with the lowest validation loss are restored.
    history = mod.fit(x_train, y_train, epochs=nbepochs, batch_size=10, validation_data=(x_val, y_val), callbacks=[callback])
    return history

## The dataset(s)

Configure here, which data set you actually want to use. In theory the notebook should work with both following db's
- [flower db](https://www.tensorflow.org/datasets/catalog/tf_flowers) 3670 colour images of 5 classes of flowers in different sizes/resolutions.
- [cifar-10](https://www.cs.toronto.edu/~kriz/cifar.html) 60000 32x32 (only) colour images in 10 classes, with 6000 images per class

(Cifar-10 will be faster to train the cnn's as the img resolution is smaller)

In [ ]:
db_2_use='cifar-10' 
#db_2_use='flowers'

### Flowers

In [ ]:
if db_2_use=='flowers' :
    dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
    data_dir = keras.utils.get_file(origin=dataset_url,
                                   fname='flower_photos',
                                   untar=True)
    data_dir = pathlib.Path(data_dir).joinpath("flower_photos")
    image_count = len(list(data_dir.glob('*/*.jpg')))
    print(image_count)

In [ ]:
if db_2_use=='flowers' :
    def dataset_to_numpy_ints(dataset):
        x = []
        y = []
    
        # Iterate through the dataset
        for image, label in dataset:
            # Rescale images from [0, 1] to [0, 255] and convert to uint8
            image_uint8 = (image.numpy() ).astype(np.uint8)
            x.append(image_uint8)  # Append the converted images
            y.append(label)  # Append labels directly
        x = np.concatenate(x, axis=0)
        y = np.concatenate(y, axis=0)
        return x, y

In [ ]:
if db_2_use=='flowers' :
    # the images have different sizes. Neural nets require always the same input size for each sample.
    # so we will rescale all img's
    img_height = 224 
    img_width = 224
    
    train_ds = keras.utils.image_dataset_from_directory(
      data_dir,
      validation_split=0.2,
      subset="training",
      seed=123,
      image_size=(img_height, img_width),
      )

    val_ds = keras.utils.image_dataset_from_directory(
      data_dir,
      validation_split=0.2,
      subset="validation",
      seed=123,
      image_size=(img_height, img_width),
      )

    class_names = train_ds.class_names

    # we want to reformat our data from 'data sets' to 'arrays'
    # at the same moment we also rescale the flower images to 0->256 (such that we have the same format for cifar-10 as for the flowers)
    x_train, y_train = dataset_to_numpy_ints(train_ds)
    x_val, y_val     = dataset_to_numpy_ints(val_ds)

### CIFAR-10

In [ ]:
if db_2_use=='cifar-10' :
    from tensorflow.keras.datasets import cifar10

    # Load CIFAR-10 data
    (x_train_full, y_train_full), (x_val_full, y_val_full) = cifar10.load_data()
    print('train:'+str(x_train_full.shape))
    print('val:'+str(x_val_full.shape))

In [ ]:
if db_2_use=='cifar-10' :
    reduce_data_set= True  # if not...my laptop runs out of memory
    if reduce_data_set :   
       # Create a random subset of the training data, as we want to limit the size of the training data to limit the training time.
       # note: reducing the size of the training data is something AI engineers will not do in practice :-)
       #np.random.seed(42)
       #np.random.seed(16)
       np.random.seed(8)
    
       indices = np.random.choice(range(x_train_full.shape[0]), 5000, replace=False)
       x_train = x_train_full[indices]
       y_train = y_train_full[indices]
    
       # Create a random subset of the validation data
       val_indices = np.random.choice(range(x_val_full.shape[0]), 1000, replace=False)
       x_val = x_val_full[val_indices]
       y_val = y_val_full[val_indices]
    else:
       x_train = x_train_full
       y_train = y_train_full
       x_val = x_val_full
       y_val = y_val_full

For CIFAR-10 this image shows the evolution in performance reached on classification of cifar-10 images over the last decade.
<center><img src="images/perf_evolution_cifar_10.png" style="background-color:white; max-width:50%"/></center>

In [ ]:
if db_2_use=='cifar-10' :
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                   'dog', 'frog', 'horse', 'ship', 'truck']
    img_width  = x_train[0].shape[0] #32
    img_height = x_train[0].shape[1] #32

    # Flatten the labels for compatibility with loss functions
    y_train = y_train.flatten()
    y_val   = y_val.flatten()

### Data general

❓ Inspection of our data
- How many training samples dow we have? 
- How many validation samples? 
- What is the resolution of our images?
- How many classes do we have in our dataset? 

In [ ]:
nb_train_samples = x_train.shape[0]
nb_val_samples = x_val.shape[0]
num_classes = len(class_names)
resolution = (x_train.shape[1], x_train.shape[2], x_train.shape[3])

In [ ]:
# do not change this cell
print({
    "nb_train_samples": nb_train_samples,
    "nb_val_samples": nb_val_samples,
    "resolution": resolution,
    "nb_classes": num_classes    
})

In [ ]:
# which classes are found in this db?
print(class_names)

In [ ]:
# what is the range of the pixel values in this db?
max_value = np.amax(x_train)
min_value = np.amin(x_train)
print("Maximum value:", max_value)
print("Minimum value:", min_value)
scale=max_value # the scale that will be used to normalise the data before the data is fed into the network

**❓Visual inspection of some samples.** (code)

 Plot the first (6) samples from the training set.
 You can use plt.imshow() to do that. Ideally you also plot the classname of the sample in the title or alike. 

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(x_train[i].astype("uint8"))
    label = int(y_train[i])
    plt.title(class_names[label])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Pixel values as features. 
Lets try to classify using a shallow neural net, with pixel values as input.
The neural net we will design is similar as in the previous notebook, be it a bit bigger.
It will use the pixel values directly as input. (this time colored images iso grayscale).
Note :The task at hand is more difficult than the digits form the notebook on neural nets.

In [ ]:
# lets ensure that we always reproduce the same results...
def set_seeds():
    np.random.seed(32)
    keras.utils.set_random_seed(32)
    tf.random.set_seed(32)
    tf.config.experimental.enable_op_determinism()
set_seeds()

**❓Design a shallow neural net similar to what we have used in the notebook on neural nets** (code)

Use an input layer which accepts a shape equal to the resolution of the images we have.
Use 3 layers of 256 neurons, followed by 1 with 64, and 1 with 32. (activation = relu)
End the network with a layer that has a number of neurons equal to the number of classes...and activation 'softmax'

In [ ]:
model = keras.models.Sequential([
  keras.layers.Input(shape=(img_width, img_height, 3)),
  keras.layers.Flatten(),
  keras.layers.Dense(256, activation="relu"),
  keras.layers.Dense(256, activation="relu"),
  keras.layers.Dense(256, activation="relu"),
  keras.layers.Dense(64, activation="relu"),
  keras.layers.Dense(32, activation="relu"),
  keras.layers.Dense(num_classes, activation="softmax")
])

In [ ]:
if db_2_use=='flowers' :
    num_epochs=20
    pat= 3
elif db_2_use=='cifar-10' :
    num_epochs=40
    pat= 10

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience=pat,restore_best_weights=True) # If the validation loss rises for 3 epochs training is stopped..and the weigths of the model with the lowest validation loss are restored.
history = model.fit(x_train, y_train, epochs=num_epochs, batch_size=10, validation_data=(x_val, y_val), callbacks=callback)

In [ ]:
plotTrainingHistory(history)

In [ ]:
y_val_pred=np.argmax(model.predict(x_val),axis=1)
displayPerformanceFigures(y_val,y_val_pred,'val samples',includeCF=False)

**❓Describe the performance. Compare with the accuracy of a lazy/random classifier** (text)

The validation accuracy printed above is the useful measure of how well the model generalizes to unseen images. A uniform random guess has an expected accuracy of 1/10 (10%) for CIFAR-10 and 1/5 (20%) for the five flower classes. Compare the measured validation accuracy with the matching baseline; the training accuracy alone can be higher because the model may overfit.

## Engineered features : e.g. HOG & color histogram

In [ ]:
from skimage.feature import hog

def color_histogram(image, bins=8):
    features = []

    for channel in range(3):
        hist, _ = np.histogram(
            image[:, :, channel],
            bins=bins,
            range=(0, 255),
            density=True
        )
        features.extend(hist)

    return np.array(features)


def extract_features(image):

    hog_features = hog(
        image,
        orientations=9,
        pixels_per_cell=(4, 4),
        cells_per_block=(2, 2),
        channel_axis=-1
    )

    color_features = color_histogram(image)

    return np.concatenate([
        hog_features,
        color_features
    ])

In [ ]:
# lets extract the (engineered) features
x_train_features = np.array([
    extract_features(image)
    for image in x_train
])

x_val_features = np.array([
    extract_features(image)
    for image in x_val
])

In [ ]:
num_features = x_train_features.shape[1]

model_for_engineered_features = keras.models.Sequential([
    keras.layers.Input(shape=(num_features,)),

    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(32, activation='relu'),

    keras.layers.Dense(num_classes, activation='softmax')
])

In [ ]:
model_for_engineered_features.summary()

In [ ]:
model_for_engineered_features.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience=pat,restore_best_weights=True) # If the validation loss rises for 3 epochs training is stopped..and the weigths of the model with the lowest validation loss are restored.
history = model_for_engineered_features.fit(x_train_features, y_train, epochs=num_epochs, batch_size=10, validation_data=(x_val_features, y_val), callbacks=callback)

In [ ]:
#lets plot the training history for the model trained on engineered features
plotTrainingHistory(history)

In [ ]:
y_val_pred=np.argmax(model_for_engineered_features.predict(x_val_features),axis=1)
displayPerformanceFigures(y_val,y_val_pred,'val samples',includeCF=False)

**❓Describe the performance. Compare with the accuracy of the shallow neural net with pixels as features** (text)

Compare the validation accuracy printed above with the pixel-only model's validation accuracy in the previous section. HOG features summarize local edge and shape patterns, while the color histograms summarize color distribution. These hand-engineered features can make useful visual structure easier for a small classifier to learn than raw pixel values, though the measured result depends on the data subset and training run.

## Self made convolutional neural networks

Convolutional networks, start from the same principle. First extract features, then classify the features.
The big difference with handcrafted/engineered features is that CNN's can find themselves (using a special type of neural nets) which feature set is most performant to do the classification job at hand. You no longer have to handcraft(potentially using tools) the features your self :-)

<center><img src="images/cnn.ppm" style="background-color:white; max-width:50%"/></center>

Convolutional neural networks, also known as CNN's, learn the values for the filters in the network.

The following snippet:

```Python

import tensorflow as tf

convLayers = keras.Sequential([
    keras.layers.Conv2D(filters=256, kernel_size=(3,3), padding="same"),
    keras.layers.Conv2D(filters=128, kernel_size=(3,3), padding="same"),
    keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding="same"),
])
```

Shows the basic building block of a CNN : the Conv2D layer.

Lets assume our input image has a resolution 224x224x3.

This neural network has 3 layers. 

The first one is a convolution layer that contains 256 filters. Each of this filter scans the entire image. Each filter is of size 3x3x3 (because we have 3 color channels). ```padding="same"``` means that after applying the convolution we want our output image to be the same size. **After applying this convolution our image size is 224x224x256.** 256 **unique** filters scanned through your image and thus **produced 256 (gray_scale) variations of your input image.** (you also could say that your 3 channel image got transformed in an image with 256 cahnnels ) 

The second layer has 128 filters. Each of these scans the image again. These filters are of size 3x3x128. The ouput here is 250x250x128, 128 unique filters scanned the image.
   
A final convolution layer scans the image. The **feature map**'s size is 32 here. 32 features are considered.

#### Feature Extraction
Features refer to the learned weights of the filters in the convolutional layers. These weights determine what patterns the filters are looking for in the input image.

**Each filter is looking for different features.** Some of them are looking for vertical lines, others horizontal lines, etc. 

Feature activations (or feature maps) are the outputs of a convolutional layer after the image has been processed by the filters. These activations represent the response of each filter to different regions of the input image, highlighting different aspects of the original image.

In summary, through successive layers of convolutions, CNNs transform the input image into a series of feature maps, each highlighting different aspects of the original image, ultimately allowing the network to recognize and interpret the high-level features in the image.

<center><img src="images/conv layer features.png" style="background-color:white; max-width:50%"/></center>

In [ ]:
x_res=224               # in cifar-10 it would be 32
y_res=224               # in cifar-10 it would be 32
chans=3                 # color
imaginary_nb_of_imgs=64 # does not matter. It is only used to build a fake/random batch of
                        # input imgs to allow us to inspect the architecture of our model.

color_imgs = np.random.rand(imaginary_nb_of_imgs,x_res,y_res,chans)

l1  = keras.layers.Conv2D(filters=256, kernel_size=(3,3), padding='same',activation='relu')
ol1 = l1(color_imgs)
print('layer 1 output:',ol1.shape)
weights_l1, biases_l1 = l1.get_weights()
print('layer 1 kernel size:',weights_l1.shape)
print()

l2 = keras.layers.Conv2D(filters=128, kernel_size=(3,3), padding='same',activation='relu')
ol2=l2(ol1)
print('layer 2 output:',ol2.shape)
weights_l2, biases_l2 = l2.get_weights()
print('layer 2 kernel size:',weights_l2.shape)
print()

l3=keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same',activation='relu')
features=l3(ol2)
print('layer 3 output:',features.shape)
weights_l3, biases_l3 = l3.get_weights()
print('layer 3 kernel size:',weights_l3.shape)

Look at the actual kernel size......for layer 1 it is not 3x3 as specified but 3x3x3 (and then 256 kernels). For layer 2 its 3x3x256 (and tyen 128 kernels), etc..
This is because in Keras, each kernel works on 3 different channels to create a single Dimensional output.  

**❓What do we still need extra in the network before we can call it it classifier model?** (text)

The convolutional layers extract image features, but a classifier also needs a classification head: flatten or otherwise aggregate the feature maps, then use dense layers to map them to one score per class. A softmax output converts those scores to class probabilities.

In [ ]:
classifier = keras.Sequential([
    keras.layers.Input(shape=(img_width, img_height,3)),
    keras.layers.Conv2D(filters=256, kernel_size=(3,3), padding="same"),
    keras.layers.Conv2D(filters=128, kernel_size=(3,3), padding="same"),
    keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding="same"),
    
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(units=num_classes, activation='softmax')
])

**❓ What do the last 4 layers do?** (text)

Flatten turns the convolutional feature maps into a vector. The next dense layers learn combinations of those features. The final dense layer has one output per class, and softmax converts its outputs into probabilities that sum to 1.

**❓ How big is this network?** (code)

Use the .summary() method to investigate the number of parameters to train this network.

In [ ]:
classifier.summary()

**❓What do you notice?** (text)

The summary shows that this model has a very large number of parameters, especially in the dense layers after Flatten. Flatten passes every location in every feature map to the dense layer, creating a large weight matrix. Pooling can shrink the feature maps before flattening and reduce that parameter count.

We need one last ingredient to make a convolutional neural network: **pooling**. The point of pooling is to downsample, it more specifically reduces the size of the images. This makes neural networks more invariant to translation (shifting an image to the left/right/up/down), makes it more computationally efficient to train (less parameters) and generally seem to improve performance.
 
<center><img src="images/pooling1.png" style="background-color:white; max-width:50%"/></center>

<center><img src="images/pooling2.png" style="background-color:white; max-width:50%"/></center>

#### Building the network

Note that we also better normalise the pixel values before feeding them to the network. So we add a rescaling layer.

For info on the layers used see : keras manuals
 - rescaling [see here](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Rescaling)
 - conv2D [see here](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D)
 - maxPooling2D [see here](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D)
 - model.fit|predict [see](https://www.tensorflow.org/api_docs/python/tf/keras/Model)

In [ ]:
my_kernel_size=3
layer_filters=[256,128,32]
model = keras.Sequential([
  keras.layers.Input(shape=(img_width, img_height,3)),
  
  keras.layers.Rescaling(1./scale, offset=-1),

  keras.layers.Conv2D(filters=layer_filters[0],kernel_size=my_kernel_size, padding='same',activation='relu'),
  keras.layers.MaxPooling2D(padding='same'), 
    

  keras.layers.Conv2D(filters=layer_filters[1], kernel_size=my_kernel_size, padding='same',activation='relu'),
  keras.layers.MaxPooling2D(padding='same'),
  
  keras.layers.Conv2D(filters=layer_filters[2], kernel_size=my_kernel_size, padding='same',activation='relu'),
  keras.layers.MaxPooling2D(padding='same'),

  keras.layers.Flatten(),

  keras.layers.Dense(128, activation='relu'),
  keras.layers.Dense(64, activation='relu'),
  #keras.layers.Dense(num_classes)
  keras.layers.Dense(units=num_classes, activation='softmax')
])

**❓ Look again at the architecture and size of the network.** (code) 

In [ ]:
model.summary()

**❓What happened to the amount of trainable parameters?** (txt)

Pooling downsamples each feature map before Flatten, so the dense layer receives a much shorter vector. This sharply reduces the number of trainable parameters and computation while retaining prominent local features.

Let's go ahead and train the classifier.
Note that we first lower the number of kernels used significantly to make training a feasible task for your machine.

In [ ]:
tf.random.set_seed(42)  # to ensure cponsistent initialisation of th weigths  
my_kernel_size=3

layer_filters=[64,32,16] # when working with cifar-10 ...this one 
#layer_filters=[128,64,32] 
#layer_filters=[256,128,32] 

model = keras.Sequential([
  
  keras.layers.Input(shape=(img_width, img_height,3)),

  keras.layers.Rescaling(1./scale, offset=-1),

  keras.layers.Conv2D(filters=layer_filters[0],kernel_size=my_kernel_size, padding='same',activation='relu'),
  keras.layers.MaxPooling2D(padding='same'), 
    

  keras.layers.Conv2D(filters=layer_filters[1], kernel_size=my_kernel_size, padding='same',activation='relu'),
  keras.layers.MaxPooling2D(padding='same'),
  
  keras.layers.Conv2D(filters=layer_filters[2], kernel_size=my_kernel_size, padding='same',activation='relu'),
  keras.layers.MaxPooling2D(padding='same'),

  keras.layers.Flatten(),

  keras.layers.Dense(128, activation='relu'),
  keras.layers.Dense(64, activation='relu'),
  keras.layers.Dense(units=num_classes, activation='softmax')
])

model.summary()

In [ ]:
set_seeds()

In [ ]:
epochs_to_train=40

In [ ]:
batch_size = 64

In [ ]:
cnn_weigths_file="cnn_classifier_"+db_2_use+"_ep_"+str(epochs_to_train)+'_'+str(layer_filters)+".weights.h5"
cnn_weigths_file

In [ ]:
#load_or_train_weights='load' (in case you have trained this model before & saved the weights)
load_or_train_weights='train'
if load_or_train_weights=='load':
    model.load_weights(cnn_weigths_file, skip_mismatch=False)
else:    
    history=compile_and_train_model(model,nbepochs=25,pat=3)    

In [ ]:
if load_or_train_weights=='train':
    plotTrainingHistory(history)

In [ ]:
# do not remove this cell
y_val_pred=np.argmax(model.predict(x_val),axis=1)
displayPerformanceFigures(y_val,y_val_pred,'validation samples',includeCF=True)

**❓ Write in a few lines your impression about the performance (compare with previous networks).**(txt)

The convolutional network can learn spatial patterns such as edges and shapes directly, unlike the pixel-only network, and its validation results should be compared with both earlier models. Use the validation accuracy and macro F1 printed above for the actual comparison; do not infer performance from training accuracy alone. The workshop's example CIFAR-10 CNN reached about 0.51 validation accuracy, while its flower example reached about 0.64, but your result can differ with the dataset, subset, and run.

### Lets play a bit with the hyper parameters, the size of the kernels, the number of neurons in the layers, the number of layers, insert drop out layers... , introduce weigth decay

#### CIFAR 10
- with kernel size 6, filters in layers 32,64,64,mp epochs 25  : val acc was 0.42
- with kernel size 3, filters in layers 32,64,64,mp epochs 25  : val acc was 0.48
- with kernel size 3, filters in layers 32,mp,64,mp,64,mp epochs 100  : val acc was 0.51 (and fast)
- with kernel size 6, filters in layers 32,mp,64,mp,64,mp        : model does not compile as inner layer shrinked too much.
- with kernel size 3, filters in layers 32,mp,32,mp,64,mp        : val f1 0.52
- with kernel size 6, filters in layers 32,mp,32,mp,64,mp        : val f1 0.48
#### FLOWERS
- with kernel size 3, filters in layers 32,mp,32,mp,32,mp        : val acc was 0.64  (but slow)
- with kernel size 6, filters in layers 32,mp,32,mp,32,mp        : val acc was 0.60  (but slow)

Note: instead of manually trying and keeping track of parameter settings / performances / weigths etc...we should use a packages like optuna and or MLflow or alike.

In [ ]:
save_model_weights=False
if save_model_weights :
   model.save_weights(cnn_weigths_file, overwrite=True)

## Transfer learning

Do we need to design a new neural network architecture for each task? Do we need to retrain everytime a neural net from scratch?
A lot of research has been done about this in the past. 
https://keras.io/api/applications/ provides several neural networks we can download and use.

The models in keras applications were trained on a dataset called imagenet that contains millions of samples and 1000 distinct classes. The convolutional layers are quite general. **This is the key way to train and use very deep neural networks if you don't have a lot of data.** The only thing you are training in this case are the Dense layers. This is a far easier task.

To demonstrate this we'll use the state of the art anno 2020 for image classification (called Xception). For this architecture you do not need to understand the diagram whatsoever, it is just indicative of the fact that it is significantly more complicated to come up with yourself and implement.

<center> Xception, VGG and Resnet </center>
<center><img src="images/xception.ppm" style="background-color:white"/></center>

### Xception

We can import a model and then use existing, trained convolutional layers. Since these layers essentially learn features it can be said that the features learnt for one task (e.g., categorizing dogs vs cats) can generalize towards other tasks. **This is called transfer learning.** This will be a key technique applied in the experiment below.

We will use XCeption as base model. XCeption needs 96x96 as input...so we will need to scale up our cifar-10 images.

In [ ]:
seed=54
np.random.seed(seed)
tf.random.set_seed(seed)

In [ ]:
# note : xception needs input at least 71x71. If smaller -> the pooling will squeeze the images to much...
if db_2_use=='flowers' :
    # no reacaling needed
    new_size=224
    x_train_resized=x_train
    x_val_resized=x_val
if db_2_use=='cifar-10' :
    new_size=96
    
    x_train_resized = np.zeros((len(x_train), new_size, new_size, 3), dtype=np.uint8)
    for i in range(len(x_train)):
        x_train_resized[i] = cv.resize(x_train[i], (new_size, new_size))

    x_val_resized = np.zeros((len(x_val), new_size, new_size, 3), dtype=np.uint8)
    for i in range(len(x_val)):
        x_val_resized[i] = cv.resize(x_val[i], (new_size,new_size))

In [ ]:
xc_base_model = keras.applications.Xception(weights="imagenet", include_top=False, input_shape=(new_size,new_size,3), pooling="max") # Include top means we don't take any dense layers. 
xc_base_model.trainable = False

The basemodel (without the top layers) generates features. To turn the basemodel into a classifier, we will preceed it with a rescaler add a few classic dense layers (possibly followed by drop out layers) and a last dense layer with a softmax activation. So in fact we replace the standard top layers with our own clasifier layers.
The weigths of this last layer will still need to be trained.

In [ ]:
def make_classifier_model(base,classifier_layer_sizes,num_classes):
   model = keras.Sequential()
   model.add(keras.layers.Rescaling(1./scale, offset=-1))
   model.add(base),

   for size in classifier_layer_sizes :
       model.add(keras.layers.Dense(size, activation='relu'))
       model.add(keras.layers.Dropout(0.2))

   model.add(keras.layers.Dense(units=num_classes, activation='softmax'))
   return model

In [ ]:
layers_2_use=[32,16]

In [ ]:
xception_classifier=make_classifier_model(xc_base_model,layers_2_use,num_classes)

xception_classifier.compile(
  optimizer=keras.optimizers.Adam(learning_rate=3e-4),
  loss=keras.losses.SparseCategoricalCrossentropy(),
  metrics=['accuracy'])

xception_classifier.build(input_shape=(None,new_size,new_size,3))
xception_classifier.summary()

**❓What can you remark about the amount of trainable parameters?**  (text)

The Xception feature extractor is pretrained on ImageNet and frozen (`trainable = False`), so its weights are not updated. Only the small dense classifier head is trained. Thus the total model still contains many parameters, but far fewer trainable parameters than a model trained end to end from scratch.

Those who want can try to out comment the line base_model.trainable = False and look at the # of trainable params in that case...(do not forget to undo this change before training the net).

In [ ]:
if db_2_use=='flowers' :
    num_epochs=10
    pat= 3
elif db_2_use=='cifar-10' :
    num_epochs=25
    pat= 10

In [ ]:
xception_weigths_file="./weigths/xception_classifier_"+db_2_use+"_ep_"+str(epochs_to_train)+"_layers_"+str(layers_2_use)+"_seed_"+str(seed)+".weights.h5"
xception_weigths_file

**❓ Fit the model** (code)

Incl. early stopping, and plot the training history
Use 'restore_best_weights=True' in the early stopping callback. 

In [ ]:
callback = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=pat,
    restore_best_weights=True
)
history = xception_classifier.fit(
    x_train_resized,
    y_train,
    epochs=num_epochs,
    batch_size=32,
    validation_data=(x_val_resized, y_val),
    callbacks=[callback]
)

**❓ Plot the training history** (code)

In [ ]:
plotTrainingHistory(history)

In [ ]:
# Predict the probabilities across all classes
predictions = xception_classifier.predict(x_val_resized)
# Select the prediction with the highest probability.
y_val_pred  = np.argmax(predictions, axis=1)
displayPerformanceFigures(y_val,y_val_pred,'self tuned xception model',includeCF=True)

**❓How does the accuracy compare to what we did with the earlier models?** (text)

Compare the validation accuracy and macro F1 printed above with the pixel-only, engineered-feature, and custom-CNN results. Transfer learning often performs better because Xception starts with useful ImageNet features and only the classification head must be learned. The notebook's reference CIFAR-10 run reported macro F1 around 0.74; that is a reference, not a guaranteed result. Your scores depend on the random subset and training run.

In case you do not manage to train the network yourself : this was the result I got after training for 25 epochs on the cifar-10 db [32 16]  layers with 0.2 dropout, an f1 of 0,74.
<center><img src="images/training history xception cifar10 25 epochs 32 16 seed 54 dropout 0_2.jpg" style="background-color:white; max-width:50%"/></center>

For the flowers db and [32 16] layers, seed :54 : I reached, an f1 of 0.87 for the validation data.
<center><img src="images/training history xception flowers 25 epochs 32 16 seed 54 dropout 0_2.jpg" style="background-color:white; max-width:50%"/></center>

Note: we still see overfitting. If you have time you can 
- increase the drop out rate
- reduce further the compexity of our classifier layers
- play with the 'batch_size' (smaller batches may reduce over fitting
- use optuna to grid search trough an hyper parameter space
The primary goal of training though is to optimise the performance on validation data. The secondary goal is to reduce overfitting. ( with an overfitted model you run an higher risk that it contains some biases. )

Another step which would prpobably increase the performance of our model would be to train the model as we did above....then 'unfreeze' (part of the) the weigths of the xception model.....and 'continue' (=not restart) the training.

In [ ]:
# in case you want to save the weights for later use
# xception_classifier.save_weights(xception_weigths_file, overwrite=True)